# 🔑 GABARITO — Atividade em Grupo: Análise de Mercado de Combustíveis (SP, 2025)

> **Documento do professor / monitor.** Contém resoluções comentadas, valores de referência e respostas-modelo para as questões interpretativas.

**Como ler este gabarito**
- Cada item traz o **código de referência** (uma solução possível — outras corretas existem) com comentários objetivos.
- Os blocos `📌 O que avaliar` indicam o critério de correção e os erros mais comuns.
- Valores numéricos foram conferidos rodando o código sobre os arquivos reais.

**Pontos de atenção (validação do enunciado):**
1. **Q3c** — pela letra do enunciado a maior margem é da bandeira **TAURUS**, mas ela tem amostra mínima (22 registros). Aceitar TAURUS como resposta literal, mas valorizar quem perceber a fragilidade amostral e citar **VIBRA** como a maior entre bandeiras relevantes.
2. **Q2d** — entre os 5 menores aparece **Barueri** (RMSP, não produtora de cana): ótimo gancho, contraria a tese puramente geográfica.
3. **Q3b** — margem mínima é **negativa** (preço de venda < distribuição em alguns registros). É esperado, não é erro de código.


In [1]:
import pandas as pd

---
## Questão 1 — Conhecendo os Dados _(2,0 pontos)_

### 1a) _(0,5 pt)_ — Carregar e descrever o dataset

In [2]:
# sep=";" porque as colunas são separadas por ponto-e-vírgula;
# decimal="," porque os preços usam vírgula como separador decimal (padrão brasileiro).
postos = pd.read_csv("postos_sp_2025.csv", sep=";", decimal=",")

# (i) shape retorna a tupla (n_linhas, n_colunas)
print("Linhas e colunas:", postos.shape)

# (ii) lista com os nomes das colunas
print("Colunas:", list(postos.columns))

Linhas e colunas: (76934, 9)
Colunas: ['Regiao', 'Estado', 'Municipio', 'Revenda', 'CNPJ', 'Produto', 'Data da Coleta', 'Valor de Venda', 'Bandeira']


**Saída esperada:** `(76934, 9)` — colunas: `Regiao, Estado, Municipio, Revenda, CNPJ, Produto, Data da Coleta, Valor de Venda, Bandeira`.

📌 *O que avaliar:* uso correto de `sep` **e** `decimal`. Sem `decimal=","`, a coluna `Valor de Venda` viria como texto e quebraria as questões seguintes.

### 1b) _(0,5 pt)_ — Registros por produto

In [3]:
# value_counts() conta quantas vezes cada categoria aparece na coluna.
postos["Produto"].value_counts()

Produto
GASOLINA      29072
ETANOL        28770
DIESEL S10    19092
Name: count, dtype: int64

**Saída esperada:** `GASOLINA 29072`, `ETANOL 28770`, `DIESEL S10 19092`.

📌 *Aceitar também* `groupby("Produto").size()`.

### 1c) _(1,0 pt)_ — Mín, média e máx da Gasolina em uma operação

In [4]:
# Filtra só GASOLINA e aplica agg() com uma lista de funções -> tudo de uma vez.
gasolina = postos[postos["Produto"] == "GASOLINA"]
gasolina["Valor de Venda"].agg(["min", "mean", "max"]).round(2)

min     4.99
mean    6.12
max     8.99
Name: Valor de Venda, dtype: float64

**Saída esperada:** `min 4.99`, `mean 6.12`, `max 8.99`.

📌 *O que avaliar:* exigir **uma única operação** (`.agg([...])`). Três chamadas separadas (`.min()`, `.mean()`, `.max()`) não atendem ao enunciado — descontar parcialmente.

---
## Questão 2 — Álcool ou Gasolina? _(4,0 pontos)_

### 2a) _(0,5 pt)_ — Criar a coluna `Mes` (YYYY-MM)

In [5]:
# Abordagem por fatiamento de texto: a data é "DD/MM/AAAA".
#   .str[6:10] -> ano (posições 6 a 9)   |   .str[3:5] -> mês (posições 3 e 4)
postos["Mes"] = postos["Data da Coleta"].str[6:10] + "-" + postos["Data da Coleta"].str[3:5]

postos[["Data da Coleta", "Mes"]].head(3)

,Data da Coleta,Mes
0,01/01/2025,2025-01
1,01/01/2025,2025-01
2,01/01/2025,2025-01


**Saída esperada:** `01/01/2025 -> 2025-01` nas 3 primeiras linhas.

✅ *Abordagem alternativa igualmente correta (datetime):*
```python
postos["Mes"] = pd.to_datetime(postos["Data da Coleta"], dayfirst=True).dt.to_period("M").astype(str)
```
Verifiquei que as duas produzem resultado idêntico. Os dados cobrem `2025-01` a `2025-06`.

### 2b) _(1,5 pt)_ — Preço médio por Município, Mês e Produto (`comparacao`)

In [6]:
# Filtra os dois produtos de interesse.
ge = postos[postos["Produto"].isin(["GASOLINA", "ETANOL"])]

# pivot_table: linhas = Municipio+Mes, colunas = cada Produto vira uma coluna,
# valores = média do Valor de Venda. reset_index() devolve Municipio/Mes como colunas comuns.
comparacao = ge.pivot_table(index=["Municipio", "Mes"],
                            columns="Produto",
                            values="Valor de Venda",
                            aggfunc="mean").reset_index()
comparacao.columns.name = None   # remove o rótulo "Produto" do cabeçalho das colunas

comparacao.head()

,Municipio,Mes,ETANOL,GASOLINA
0,ADAMANTINA,2025-01,3.896250,5.935000
1,ADAMANTINA,2025-02,3.984688,6.009375
2,ADAMANTINA,2025-03,3.987500,6.078125
3,ADAMANTINA,2025-04,3.942250,6.027000
4,ADAMANTINA,2025-05,3.914516,6.051875


**Saída esperada:** DataFrame com `522 linhas × 4 colunas` (`Municipio, Mes, ETANOL, GASOLINA`).
Primeira linha: `ADAMANTINA | 2025-01 | 3.90 | 5.94`.

✅ *Alternativa com `groupby` + `merge`* (também correta — conferido que dá o mesmo resultado):
```python
et = ge[ge["Produto"]=="ETANOL"].groupby(["Municipio","Mes"])["Valor de Venda"].mean().rename("ETANOL")
ga = ge[ge["Produto"]=="GASOLINA"].groupby(["Municipio","Mes"])["Valor de Venda"].mean().rename("GASOLINA")
comparacao = pd.merge(et, ga, on=["Municipio","Mes"]).reset_index()
```
📌 *Erro comum:* esquecer o `reset_index()` e entregar um índice multinível em vez de 4 colunas.

### 2c) _(1,0 pt)_ — Razão e combustível vantajoso

In [7]:
# 1) Razão entre os preços médios.
comparacao["Razao"] = comparacao["ETANOL"] / comparacao["GASOLINA"]

# 2) Compara com 0.708. A comparação gera True/False, e .replace() troca por rótulos.
#    Regra: <= 0.708 -> etanol compensa.
comparacao["Combustivel_Vantajoso"] = (comparacao["Razao"] <= 0.708).replace(
    {True: "Etanol", False: "Gasolina"})

comparacao.head()

,Municipio,Mes,ETANOL,GASOLINA,Razao,Combustivel_Vantajoso
0,ADAMANTINA,2025-01,3.896250,5.935000,0.656487,Etanol
1,ADAMANTINA,2025-02,3.984688,6.009375,0.663079,Etanol
2,ADAMANTINA,2025-03,3.987500,6.078125,0.656041,Etanol
3,ADAMANTINA,2025-04,3.942250,6.027000,0.654098,Etanol
4,ADAMANTINA,2025-05,3.914516,6.051875,0.646827,Etanol


**Saída esperada (5 primeiras linhas):** todas com `Razao ≈ 0.65` e `Combustivel_Vantajoso = "Etanol"`.
No total: **514** combinações favorecem o Etanol e **8** a Gasolina.

✅ *Alternativas válidas:* `.apply(lambda r: ...)` ou `np.where(...)`. Aceitar qualquer uma que produza os rótulos corretos.
📌 *Atenção ao operador:* a regra é `<=` (menor **ou igual**); usar `<` muda a classificação no limiar.

### 2d) _(1,0 pt)_ — Municípios com maior margem para aumento de preço

In [8]:
# Razão MÉDIA por município (média ao longo dos meses).
razao_media = comparacao.groupby("Municipio")["Razao"].mean()

# Mantém só onde o etanol foi vantajoso na média e pega os 5 MENORES (maior folga p/ subir preço).
top5 = razao_media[razao_media <= 0.708].sort_values().head(5).round(4)
top5

Municipio
SAO JOSE DO RIO PRETO    0.6259
ARACATUBA                0.6279
BARUERI                  0.6320
BIRIGUI                  0.6347
VOTUPORANGA              0.6391
Name: Razao, dtype: float64

**Saída esperada (menor → maior razão média):**

| Município | Razão média |
|---|---|
| SAO JOSE DO RIO PRETO | 0.6259 |
| ARACATUBA | 0.6279 |
| BARUERI | 0.6320 |
| BIRIGUI | 0.6347 |
| VOTUPORANGA | 0.6391 |

**Resposta interpretativa (modelo):**
Quatro dos cinco municípios — *São José do Rio Preto, Araçatuba, Birigui e Votuporanga* — ficam no **noroeste paulista**, coração da produção de cana-de-açúcar e etanol do estado. A proximidade das usinas reduz o **custo logístico/frete** do etanol até o posto, derrubando seu preço final; com etanol barato frente à gasolina, a razão cai bem abaixo de 0,708 e sobra margem para a distribuidora elevar o preço sem perder competitividade.

⚠️ **Exceção a destacar:** *Barueri* fica na **Região Metropolitana de SP**, não é zona produtora. Sua razão baixa sugere outros fatores (forte concorrência entre postos, perfil de mercado, mix de bandeiras). Valorizar o grupo que **percebe e comenta** essa exceção em vez de forçar a explicação geográfica para todos.

📌 *O que avaliar:* `groupby` correto, ordenação crescente, recorte `<= 0.708`, e uma justificativa que conecte preço baixo do etanol à proximidade da produção.

---
## Questão 3 — Margem Comercial dos Postos _(4,0 pontos)_

### 3a) _(0,5 pt)_ — Carregar distribuição e fazer o merge (`dados`)

In [9]:
distribuicao = pd.read_csv("distribuicao_municipios_sp_2025.csv", sep=";", decimal=",")

# Cruza preços de venda (postos) com preços de distribuição usando as 3 chaves em comum.
# A coluna "Mes" já existe em postos (criada na Q2a) e em distribuicao.
dados = pd.merge(postos, distribuicao, on=["Municipio", "Mes", "Produto"])

print("dados shape:", dados.shape)

dados shape: (76934, 11)


**Saída esperada:** `(76934, 11)` — nenhuma linha é perdida no merge (todas as combinações encontram correspondência na distribuição).

📌 *O que avaliar:* as **três** chaves (`Municipio`, `Mes`, `Produto`). Faltar `Produto`, por exemplo, multiplicaria as linhas (cruzaria gasolina com preço de etanol). Se o aluno ainda não criou `Mes`, o merge falha — depende da Q2a.

### 3b) _(1,0 pt)_ — Coluna `Margem` e estatísticas globais

In [10]:
# Margem bruta por litro = quanto o posto cobra menos quanto pagou à distribuidora.
dados["Margem"] = dados["Valor de Venda"] - dados["Preco_Distribuicao"]

dados["Margem"].agg(["min", "mean", "max"]).round(2)

min    -0.28
mean    0.74
max     3.88
Name: Margem, dtype: float64

**Saída esperada:** `min -0.28`, `mean 0.74`, `max 3.88`.

ℹ️ *Sobre a margem mínima negativa:* não é erro. Em 619 registros (de ~77 mil) o preço de venda coletado ficou abaixo do preço médio de distribuição do município/mês (promoções, defasagem temporal da média, ruído de coleta). Vale comentar com a turma.

### 3c) _(0,5 pt)_ — Margem média por bandeira (Gasolina)

In [11]:
# Só gasolina, média da margem por bandeira, ordenada do maior para o menor.
gas = dados[dados["Produto"] == "GASOLINA"]
margem_bandeira = gas.groupby("Bandeira")["Margem"].mean().sort_values(ascending=False)
margem_bandeira.round(2)

Bandeira
TAURUS              1.06
VIBRA               0.98
IPIRANGA            0.85
RAIZEN              0.84
SIM DISTRIBUIDOR    0.69
TOTALENERGIES       0.68
ALE                 0.66
D`MAIS              0.54
BRANCA              0.54
RODOIL              0.26
Name: Margem, dtype: float64

**Saída esperada (maior → menor, R$/L):** `TAURUS 1.06`, `VIBRA 0.98`, `IPIRANGA 0.85`, `RAIZEN 0.84`, … , `BRANCA 0.54`, `RODOIL 0.26`.

**Resposta:** pela letra do enunciado, a maior margem média é da bandeira **TAURUS**.

⚠️ **Cuidado na correção:** TAURUS tem apenas **22 registros** de gasolina no dataset — amostra pequena e pouco confiável. Entre as bandeiras com volume relevante, **VIBRA** lidera (R$ 0,98/L). Aceitar TAURUS como resposta literal, **mas dar crédito extra** a quem nota a fragilidade amostral. (Esse é o ponto de validação nº 1 do topo do gabarito.)

### 3d) _(1,5 pt)_ — Decisão de investimento: VIBRA vs BRANCA

In [12]:
# (1) Diferença de margem por litro entre VIBRA e BRANCA (reaproveita a série da 3c).
dif_litro = margem_bandeira["VIBRA"] - margem_bandeira["BRANCA"]
print("Diferença de margem por litro (VIBRA - BRANCA): R$", round(dif_litro, 4))

# (2) Projeção mensal para um posto que vende 50.000 L de gasolina/mês.
dif_mensal = dif_litro * 50000
print("Diferença estimada de margem mensal: R$", round(dif_mensal, 2))

Diferença de margem por litro (VIBRA - BRANCA): R$ 0.4435
Diferença estimada de margem mensal: R$ 22175.43


**Saída esperada:** diferença por litro ≈ **R$ 0,4435**; diferença mensal ≈ **R$ 22.175,43**.

**Resposta interpretativa (modelo — item 3):**
A franquia VIBRA gera ~R$ 22 mil/mês a mais de margem bruta de gasolina que a bandeira branca. **Mas a margem maior, sozinha, não decide.** Os dados **não capturam**, entre outros:
- **Custos da franquia:** taxa de adesão, royalties, aluguel de marca, exigências de padronização/reforma e contrato de exclusividade de compra.
- **Volume e mix:** a projeção fixa 50.000 L; bandeira e localização afetam o volume real e a venda de etanol/diesel, loja de conveniência e outros serviços.
- **Risco e poder de barganha:** o posto branco compra de quem oferecer melhor preço; o franqueado fica preso ao fornecedor, mas ganha marketing, crédito e fluxo de clientes da marca.
- **Investimento inicial e prazo de retorno**, condições de financiamento e fatores locais (concorrência, trânsito, demanda).

Conclusão esperada: a decisão exige comparar a margem adicional **líquida dos custos da franquia**, não a margem bruta isolada.

📌 *O que avaliar:* contas corretas (itens 1 e 2) + ao menos 2–3 fatores não capturados pelos dados, com raciocínio coerente.